# Apprentissage non supervisé: Partie 1 - Analyse d'un dataset de consommateurs

In [ ]:
import pandas as pd
data=pd.read_excel("CustomerDataSet.xls")
data.shape

#### 1. Afficher quelques attributs du dataset

In [ ]:
# Afficher les premières lignes
print(data.head())

# Afficher les types des colonnes
print(data.dtypes)

# Statistiques descriptives
print(data.describe())

#### 2. Quels sont les attributs qu'on peut utiliser pour le clustering ?

In [ ]:
# Les attributs numériques continus utilisables pour le clustering sont :
# - ItemsBought   : nombre d'articles achetés
# - ItemsReturned : nombre d'articles retournés
#
# On exclut :
# - CustomerID : simple identifiant, sans signification pour le clustering
# - ZipCode    : code géographique catégoriel
# - Product    : identifiant de produit

print("Attributs utilisables pour le clustering : ItemsBought, ItemsReturned")
print(data[['ItemsBought', 'ItemsReturned']])

#### 3. Regrouper l'ensemble de données en utilisant K-Means. 
- Utiliser les deux attributs `ItemsBought` et `ItemsReturned`
- Visualiser le regroupement en utilisant `ItemsBought` et `ItemsReturned` pour les axes x et y, et l'identifiant du cluster pour la couleur des points de données.
- Répétez le regroupement pour différentes valeurs de K.

In [ ]:
# importer KMeans
from sklearn.cluster import KMeans

# importer matplotlib
import matplotlib.pyplot as plt

# importer preprocessing (StandardScaler pour la normalisation)
from sklearn import preprocessing

In [ ]:
# créer le StandardScaler
scaler = preprocessing.StandardScaler()

# Copiez le dataframe avant le prétraitement afin que nous
# puissions accéder aux valeurs d'origine ultérieurement.
df2 = data.copy()

# préparer les deux attributs par le StandardScaler
df2[['ItemsBought', 'ItemsReturned']] = scaler.fit_transform(
    df2[['ItemsBought', 'ItemsReturned']]
)

# setup a figure
plt.figure(figsize=(14, 10))
ks = [2, 3, 4, 5, 6, 7]

# itérer sur la liste des valeurs de k
for idx, i in enumerate(ks, start=1):
    # créer le clusterer
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)

    # créer le regroupement
    labels = kmeans.fit_predict(df2[['ItemsBought', 'ItemsReturned']])

    # ajouter au subplot
    plt.subplot(3, 2, idx)
    plt.tight_layout(pad=3.0)

    # les étiquettes
    plt.title("#clusters (K) = {}".format(i))
    plt.xlabel('ItemsBought')
    plt.ylabel('ItemsReturned')

    # créer le scatter
    plt.scatter(df2['ItemsBought'], df2['ItemsReturned'],
                c=labels, cmap='tab10', s=80, edgecolors='k', linewidths=0.5)

# Afficher la figure
plt.show()

In [ ]:
df2.head()

#### 5. Quelle(s) valeur(s) K ont du sens et comment étiquetez-vous les clusters résultants ?

In [ ]:
# K = 3 est la valeur la plus pertinente.
# On peut clairement identifier 3 profils de clients :
#
#   Cluster 0 : Clients qui achètent beaucoup et retournent peu
#               → Bons clients fidèles
#
#   Cluster 1 : Clients avec achats et retours modérés
#               → Clients réguliers
#
#   Cluster 2 : Clients qui achètent peu mais retournent beaucoup
#               → Clients insatisfaits / à risque

print("Valeur K retenue : K = 3")
print("Cluster 0 : Bons clients fidèles (beaucoup d'achats, peu de retours)")
print("Cluster 1 : Clients réguliers (achats et retours moyens)")
print("Cluster 2 : Clients insatisfaits (peu d'achats, beaucoup de retours)")

#### 6. Ajouter une colonne contenant les étiquettes de classes à votre DataFrame

In [ ]:
# Créer KMeans avec K = 3
kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10)

# Ajouter la colonne 'Cluster' au DataFrame original
data['Cluster'] = kmeans3.fit_predict(df2[['ItemsBought', 'ItemsReturned']])

# Ajouter aussi à df2 (normalisé) pour les graphiques suivants
df2['Cluster'] = data['Cluster']

# Afficher le résultat
print(data[['Customer ID', 'ItemsBought', 'ItemsReturned', 'Cluster']])

#####  7. Effectuez à nouveau le regroupement avec `K=3`. Ajoutez les identifiants de produits `Product` au graphique à l'aide de la fonction `annotate()` et interprétez les résultats.



In [ ]:
# Créer KMeans avec K = 3
kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=10)

# Créer le clustering
labels = kmeans3.fit_predict(df2[['ItemsBought', 'ItemsReturned']])

# Créer un scatter
plt.figure(figsize=(9, 7))
plt.scatter(df2['ItemsBought'], df2['ItemsReturned'],
            c=labels, cmap='tab10', s=100, edgecolors='k', linewidths=0.6)

# annoter les produits en utilisant la colonne 'Product' et la fonction annotate
for i, row in data.iterrows():
    plt.annotate(str(row['Product']),
                 (df2.loc[i, 'ItemsBought'], df2.loc[i, 'ItemsReturned']),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

# les étiquettes des axes
plt.xlabel('ItemsBought')
plt.ylabel('ItemsReturned')
plt.title('K-Means K=3 Annotation par Product ID')

# Afficher
plt.show()

# Interprétation :
# - Les produits du cluster avec beaucoup d'achats et peu de retours (ex: 1343, 1365)
#   sont des produits populaires et satisfaisants
# - Les produits avec beaucoup de retours nécessitent une révision
#   (qualité ou description produit à améliorer)

#####  8.  Intérpréter le regroupement en utilisant le code zip.

- Pour répondre à cette question,  en retrace le graphiqe cette fois en annotant les clusters avec le code postal des clients. Notez que nous pouvons utiliser l'ensemble de données original au lieu de celui prétraité.

In [ ]:
# Créer le scatter (données originales non normalisées)
plt.figure(figsize=(9, 7))
plt.scatter(data['ItemsBought'], data['ItemsReturned'],
            c=data['Cluster'], cmap='tab10', s=100, edgecolors='k', linewidths=0.6)

# Annoter chaque point de données en utilisant le code ZIP
for i, row in data.iterrows():
    plt.annotate(str(row['ZipCode']),
                 (row['ItemsBought'], row['ItemsReturned']),
                 textcoords="offset points", xytext=(5, 5), fontsize=8, color='darkblue')

# les étiquettes
plt.xlabel('ItemsBought')
plt.ylabel('ItemsReturned')
plt.title('K-Means K=3 Annotation par ZipCode')

# Afficher le plot
plt.show()

# Interprétation :
# - Les clients des zones ZipCode 1 et 2 achètent beaucoup et retournent peu
#   → Zones géographiques de bons clients
# - Les clients du ZipCode 5 retournent davantage d'articles
#   → Peut indiquer une insatisfaction ou un problème de livraison dans cette région

#### 9 Regroupez l'ensemble de données en utilisant la classification ascendante hiérarchique (Agglomerative Hierarchical Clustering).

- Pour tracer un dendrogramme, nous devons utiliser la fonction `linkage()` de la bibliothèque `scipy` plutôt que le regroupeur de la bibliothèque `scikit-learn`. Après avoir créé le regroupement avec la fonction `linkage()`, nous pouvons le tracer en utilisant la fonction `dendrogram()`.

In [ ]:
# importer linkage et dendrogram from scipy
from scipy.cluster.hierarchy import linkage, dendrogram

# Créer le clustering
Z = linkage(df2[['ItemsBought', 'ItemsReturned']], method='ward')

# Afficher le dendrogram
plt.figure(figsize=(14, 6))
dendrogram(Z, labels=data['Customer ID'].tolist())

# Les étiquettes
plt.xlabel('Customer IDs')
plt.ylabel('distance')
plt.title('Dendrogramme Classification Hiérarchique (Ward)')

# Afficher
plt.show()

# Interprétation :
# - Chaque fusion dans l'arbre représente un regroupement de clients similaires
# - La hauteur d'une fusion indique la distance entre les groupes fusionnés
# - Une grande hauteur = groupes très différents
# - On peut couper l'arbre à différents niveaux pour obtenir N clusters

#### 10. Trnquer la classification hiérarchique de manière à obtenir 3 ou 4 groupes de clients. 


In [ ]:
from sklearn.cluster import AgglomerativeClustering

# setup a figure
plt.figure(figsize=(14, 10))

# itérer sur les différents nombres de clusters (ici: 3 et 4)
counter = 1
for i in [3, 4]:

    # ajouter un subplot
    plt.subplot(2, 2, counter)
    counter += 1

    # Les étiquettes
    plt.tight_layout(pad=3.0)
    plt.title('Dendrogram - {} clusters'.format(i))
    plt.xlabel('Count of Customers')
    plt.ylabel('distance')

    # Créer le clustering et afficher le dendrogram (tronqué)
    dendrogram(Z, truncate_mode='lastp', p=i, show_contracted=True)

    # Ajouter un second subplot (scatter)
    plt.subplot(2, 2, counter)
    counter += 1

    # Créer le clustering par troncature du dendrogram
    agg = AgglomerativeClustering(n_clusters=i)
    agg_labels = agg.fit_predict(df2[['ItemsBought', 'ItemsReturned']])

    plt.tight_layout(pad=3.0)
    plt.title('Regroupement hiérarchique {} clusters'.format(i))
    plt.xlabel('ItemsBought')
    plt.ylabel('ItemsReturned')
    plt.scatter(data['ItemsBought'], data['ItemsReturned'],
                c=agg_labels, cmap='tab10', s=100, edgecolors='k', linewidths=0.6)

# Afficher la figure
plt.show()

# Interprétation :
# - Avec 3 clusters : résultat cohérent avec K-Means (K=3), 3 profils distincts
# - Avec 4 clusters : segmentation plus fine,
#   un groupe se subdivise en deux sous-profils

# Partie 2 : Analyse d'un dataset d'étudiant  


#### 1.  Agréger le jeu de données des étudiants par étudiant et calculez la moyenne des notes ainsi que la moyenne du nombre de cours suivis. Utiliser la fonction `groupby()`

In [ ]:
# Charger le fichier StudentData.xls dans un DataFrame
sdata = pd.read_excel("StudentData.xls")

# Afficher les premiers enregistrements
sdata.head()

In [ ]:
# Grouper le dataframe par étudiant (le nom de l'étudiant) et calculer les valeurs moyennes
student_agg = sdata.groupby('Name')[['Mark', 'Attended']].mean().reset_index()
student_agg.columns = ['Name', 'AvgMark', 'AvgAttended']

# Afficher les premiers enregistrements
student_agg.head(10)

In [ ]:
print(student_agg)

#### 2. Utiliser K-Means pour créer un regroupement. 

- Exécutez un regroupeur KMeans sur les données et tracez-le dans un graphique de dispersion (scatter plot). Il est judicieux d'annoter les points de données avec les noms des étudiants.

In [ ]:
# Envisager un prétraitement (normalisation)
scaler2 = preprocessing.StandardScaler()
X_students = scaler2.fit_transform(student_agg[['AvgAttended', 'AvgMark']])

# Créer une instance de KMeans
kmeans_s = KMeans(n_clusters=3, random_state=42, n_init=10)

# Créer le clustering
student_agg['Cluster'] = kmeans_s.fit_predict(X_students)

# Afficher le scatter
plt.figure(figsize=(10, 8))
plt.scatter(student_agg['AvgAttended'], student_agg['AvgMark'],
            c=student_agg['Cluster'], cmap='Set1',
            s=120, edgecolors='k', linewidths=0.6)

# Les étiquettes
plt.xlabel("Attended classes")
plt.ylabel("Mark")
plt.title("K-Means K=3 Profils d'étudiants")

# Annoter les données par les noms des étudiants
for _, row in student_agg.iterrows():
    plt.annotate(row['Name'],
                 (row['AvgAttended'], row['AvgMark']),
                 textcoords="offset points", xytext=(5, 4), fontsize=8)

# Afficher la figure
plt.show()

# Interprétation :
# - Cluster 0 : Étudiants assidus avec bonnes notes      → Étudiants performants
# - Cluster 1 : Étudiants assidus mais notes moyennes    → Étudiants réguliers
# - Cluster 2 : Étudiants peu assidus et mauvaises notes → Étudiants en difficulté

#### 3. Regrouper le datset en uilisant le clustering hérarchique aglomératif. Utiliser différentes métriques de liaisons. 

- Nous définissons d'abord les paramètres que nous voulons tester (les différents modes de liaison) puis itérons sur ces valeurs dans une boucle for. À l'intérieur de la boucle, nous créons le regroupement avec les paramètres respectifs et traçons le dendrogramme.

In [ ]:
# lister les métriques à utiliser
modes = ['single', 'average', 'complete']

# créer une figure
plt.figure(figsize=(20, 5))
y_axis = None

# itérer sur la liste des métriques
for i, mode in enumerate(modes):

    # Créer un subplot
    y_axis = plt.subplot(1, 3, i + 1)

    # les étiquettes du subplot
    plt.title('Dendrogram - linkage mode: {}'.format(mode))
    plt.xlabel('ID of student')
    plt.ylabel('distance')

    # Créer le clustering
    Z_s = linkage(X_students, method=mode)

    # Afficher le dendrogram
    dendrogram(Z_s,
               labels=student_agg['Name'].apply(lambda x: x.split()[0]).tolist(),
               leaf_rotation=90,
               leaf_font_size=8)

# Afficher la figure
plt.tight_layout()
plt.show()

# Interprétation :
# - Single   : sensible aux outliers, tend à créer des chaînes
# - Average  : compromis équilibré
# - Complete : clusters plus compacts et homogènes → méthode recommandée ici